# 03 — Inferencia Multivariable y Evaluación de Supuestos

## 1. Marco Teórico
En esta sección comparamos formalmente el vector de medias ambientales (centroides) correspondientes a los sitios con Baja Diversidad de Aves ($Q1$ de Shannon) frente a los de Alta Diversidad ($Q3$). El objetivo es determinar si las condiciones ambientales medias difieren significativamente entre ambos escenarios ecológicos.

### Supuesto 1: Normalidad Multivariante (Prueba de Mardia)
La prueba de Mardia evalúa la normalidad multivariante contrastando los coeficientes de asimetría ($b_{1,p}$) y curtosis ($b_{2,p}$) multivariantes sobre los vectores de datos $\mathbf{x}_i$.
Las hipótesis son:
$$H_0: \text{Los datos provienen de una población con distribución normal multivariada}$$
$$H_1: \text{Los datos no siguen una distribución normal multivariada}$$

La asimetría de Mardia se define como:
$$b_{1,p} = \frac{1}{n^2} \sum_{i=1}^n \sum_{j=1}^n \left( (\mathbf{x}_i - \bar{\mathbf{x}})^T \mathbf{S}^{-1} (\mathbf{x}_j - \bar{\mathbf{x}}) \right)^3$$
Para muestras grandes, el estadístico $A_1 = \frac{n}{6} b_{1,p}$ sigue una distribución $\chi^2_d$ con $d = \frac{p(p+1)(p+2)}{6}$ grados de libertad.

La curtosis de Mardia se define como:
$$b_{2,p} = \frac{1}{n} \sum_{i=1}^n \left( (\mathbf{x}_i - \bar{\mathbf{x}})^T \mathbf{S}^{-1} (\mathbf{x}_i - \bar{\mathbf{x}}) \right)^2$$
El estadístico estandarizado de la curtosis se comporta asintóticamente como una normal estándar: $Z = \frac{b_{2,p} - p(p+2)}{\sqrt{8p(p+2)/n}} \sim N(0, 1)$.

### Supuesto 2: Homocedasticidad Multivariante (Prueba M de Box)
La prueba M de Box evalúa la igualdad de matrices de covarianza entre $g$ grupos independientes. 
$$H_0: \boldsymbol{\Sigma}_1 = \boldsymbol{\Sigma}_2 = \dots = \boldsymbol{\Sigma}_g$$
$$H_1: \exists k, l \text{ tales que } \boldsymbol{\Sigma}_k \neq \boldsymbol{\Sigma}_l$$

El estadístico M se calcula como:
$$M = (N-g) \ln |\mathbf{S}_{pooled}| - \sum_{k=1}^g (n_k-1) \ln |\mathbf{S}_k|$$
donde $\mathbf{S}_k$ es la matriz de covarianza del grupo $k$, $\mathbf{S}_{pooled}$ es la matriz de covarianza combinada y $N = \sum n_k$. Se aplica una constante de corrección para aproximar $M$ a una distribución $\chi^2$ o a una distribución $F$ de Snedecor.

### Contraste de Centroides
1. **Estadístico $T^2$ de Hotelling:** Es la extensión multivariada de la prueba t de Student para muestras independientes. Si los supuestos de normalidad y homocedasticidad se cumplen:
   $$T^2 = \frac{n_1 n_2}{n_1 + n_2} (\bar{\mathbf{x}}_1 - \bar{\mathbf{x}}_2)^T \mathbf{S}_{pooled}^{-1} (\bar{\mathbf{x}}_1 - \bar{\mathbf{x}}_2)$$
   Bajo $H_0: \boldsymbol{\mu}_1 = \boldsymbol{\mu}_2$, el estadístico escalado se distribuye como una variable $F$:
   $$\frac{n_1 + n_2 - p - 1}{p(n_1 + n_2 - 2)} T^2 \sim F_{p, n_1+n_2-p-1}$$
2. **Prueba Permutacional de Distancia entre Centroides:** Dado que el tamaño de muestra es muy grande ($n > 100,000$), las pruebas paramétricas clásicas como Hotelling pierden robustez si hay violaciones severas de supuestos. La prueba permutacional no paramétrica es el estándar de oro. Mezcla las etiquetas de los grupos aleatoriamente en 10,000 iteraciones para construir la distribución empírica de la distancia de la norma euclidiana entre centroides bajo la hipótesis nula de igualdad. El p-valor se calcula como la proporción de permutaciones donde la distancia permutada supera a la observada.

In [1]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
os.chdir(ROOT)

import pandas as pd
import IPython.display as display
import importlib.util
spec = importlib.util.spec_from_file_location('multivar', ROOT / 'Scripts/09_multivariate_inference_assumptions.py')
multivar = importlib.util.module_from_spec(spec)
spec.loader.exec_module(multivar)

df = pd.read_parquet(multivar.DATA_PATH)
groups, low, high = multivar.prepare_groups(df)
print(f"Datos cargados para inferencia. n total = {len(groups)} (Baja Diversidad n = {len(low)}, Alta Diversidad n = {len(high)})")
print(f"Predictores ambientales evaluados: {multivar.Z_CONTINUOUS_ENV_VARS}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Miguel Camargo\\Desktop\\Parcial FInal Estadistica\\Notebooks_respaldo\\Scripts\\09_multivariate_inference_assumptions.py'

## 2. Diagnóstico de Normalidad Multivariante (Mardia)
Ejecutamos la prueba de Mardia para la muestra completa y para cada grupo por separado.

In [ ]:
# Correr pruebas y obtener tablas
mardia, box_m, hotelling, permutation, means, permuted_distances = multivar.save_results(df, groups, low, high, n_permutations=999)
print("=== RESULTADOS DE LA PRUEBA DE NORMALIDAD DE MARDIA ===")
display.display(mardia.round(4))

## 3. Diagnóstico de Homocedasticidad (M de Box)
Evaluamos si las matrices de covarianza de las condiciones ambientales de los sitios de Baja y Alta diversidad son estadísticamente idénticas.

In [ ]:
print("=== RESULTADOS DE LA PRUEBA M DE BOX ===")
display.display(box_m.round(4))

## 4. Contraste de Medias Multivariantes (Hotelling $T^2$ y Permutaciones)
Comparamos la hipótesis de igualdad de vectores de medias usando el método paramétrico clásica (Hotelling) y el método robusto no paramétrico (Prueba Permutacional de distancia de centroides con su distribución).

In [ ]:
print("=== ESTADÍSTICO T2 DE HOTELLING PARAMÉTRICO ===")
display.display(hotelling.round(4))
print("\n=== CONTRASENSIÓN PERMUTACIONAL DE DISTANCIA DE CENTROIDES ===")
display.display(permutation.round(4))

# Graficar distribución de permutación
multivar.plot_permutation_distribution(permuted_distances, permutation.loc[0, 'observed_centroid_distance'])
from IPython.display import Image
Image(filename='Figures/Multivariate_Inference/03_permutation_centroid_distance_continuous.png')

## 5. Análisis del Perfil Ambiental Medio de los Grupos
Visualizamos los perfiles medios estandarizados para ver en qué variables se dan las diferencias más grandes entre los sitios con Alta Diversidad (Q3) y Baja Diversidad (Q1). Adicionalmente, graficamos las elipses de confianza del 95% para los centroides proyectados en el plano bidimensional de Temperatura y Humedad.

In [ ]:
multivar.plot_group_profile(means)
multivar.plot_centroid_confidence_ellipses(groups)

print("=== DIFERENCIA DE MEDIAS ORDENADAS DE MANERA CRECIENTE (Alta - Baja) ===")
display.display(means)

from IPython.display import display
display(Image(filename='Figures/Multivariate_Inference/01_group_mean_profile_continuous.png'))
display(Image(filename='Figures/Multivariate_Inference/02_centroid_confidence_ellipses_temp_humidity.png'))

## Conclusiones del Contraste de Hipótesis y Supuestos
1. **Rechazo de Normalidad y Homocedasticidad:** Tanto la prueba de Mardia como la M de Box arrojan $p$-valores extremadamente bajos ($p < 0.0001$), rechazando la hipótesis de normalidad y covarianza homogénea. Esto invalida teóricamente la precisión matemática del test paramétrico de Hotelling, obligándonos a confiar en la prueba permutacional.
2. **Diferencias Significativas:** La prueba permutacional confirma con alta certidumbre que los centroides ambientales son significativamente distintos ($p = 0.001$). Las aves de alta diversidad de eBird se agrupan en perfiles ambientales particulares.
3. **Interpretación del Perfil Ecológico:** Al observar el perfil medio, los sitios de **Alta Diversidad (Q3)** muestran niveles menores de material particulado ($PM_{2.5}$ y $PM_{10}$ por debajo del promedio general, $Z < 0$) y niveles ligeramente más altos de Humedad Relativa y O3 con respecto a las áreas degradadas de Baja Diversidad (Q1). Esto sugiere que la polución del aire está fuertemente correlacionada de forma negativa con la presencia y diversidad de avifauna en el ecosistema urbano.